# AI World Cup Championship: linear notebook

This version deliberately avoids project imports and helper functions. Each cell performs one visible step, so you can inspect its inputs and outputs before continuing. Run from top to bottom. API calls can incur costs.

## 1. Install packages

Run once in a new environment, then restart the notebook kernel if necessary.

In [ ]:
# %pip install "openai>=1,<3" "tavily-python>=0.5,<1" "python-dotenv>=1,<2" "pydantic>=2.7,<3" "gradio>=5,<7"

## 2. Load `.env` and imports

Create `.env` beside this notebook. `load_dotenv()` reads it without putting secrets in the notebook.

In [ ]:
import os
import json
import sqlite3
from datetime import date, datetime, time, timedelta, timezone
from pathlib import Path
from typing import Annotated
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import gradio as gr
import httpx
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field
from tavily import TavilyClient

load_dotenv()

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY", "")
FOOTBALL_DATA_API_KEY = os.environ.get("FOOTBALL_DATA_API_KEY", "")

assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY is missing from .env"
assert TAVILY_API_KEY, "TAVILY_API_KEY is missing from .env"
assert FOOTBALL_DATA_API_KEY, "FOOTBALL_DATA_API_KEY is missing from .env"
print("OpenRouter, Tavily, and football-data.org keys loaded.")

## 3. Configuration

OpenRouter accepts the OpenAI SDK because its chat endpoint follows the OpenAI-compatible format. The `base_url` redirects SDK requests to OpenRouter.

In [ ]:
MATCH_TIMEZONE = "America/Chicago"
TODAY = datetime.now(ZoneInfo(MATCH_TIMEZONE)).date()
SELECTED_DATE = date(2026, 6, 21)  # Replace with date(2026, 6, 19) to study an earlier day.
DB_PATH = Path("data/notebook_championship.db")
FOOTBALL_DATA_URL = "https://api.football-data.org/v4"
FOOTBALL_DATA_COMPETITION = "WC"

MODELS = [
    {"id": "gpt-5.5", "model": "openai/gpt-5.5"},
    {"id": "claude-sonnet-4.6", "model": "anthropic/claude-sonnet-4.6"},
    {"id": "gemini-3.5-flash", "model": "google/gemini-3.5-flash"},
    {"id": "grok-4.3", "model": "x-ai/grok-4.3"},
    {"id": "deepseek-v3", "model": "deepseek/deepseek-chat"},
]
MASTER_MODEL = "openai/gpt-5.5"
TAVILY_SEARCH_DEPTH = "basic"
TEAM_NEWS_MAX_RESULTS = 4
BETTING_MAX_RESULTS = 5
CONTEXT_MAX_RESULTS = 3
TEAM_NEWS_CONTENT_CHARS = 1200
BETTING_CONTENT_CHARS = 1600
CONTEXT_CONTENT_CHARS = 800
RESEARCH_SUMMARY_MODEL = "deepseek/deepseek-chat"
RESEARCH_SUMMARY_MAX_TOKENS = 1100
BETTING_ALLOWED_DOMAINS = [
    "actionnetwork.com", "bet365.com", "betmgm.com", "caesars.com",
    "covers.com", "draftkings.com", "espn.com", "fanduel.com",
    "foxsports.com", "ladbrokes.com", "oddschecker.com", "oddspedia.com",
    "paddypower.com", "racingpost.com", "sports.yahoo.com",
    "thelines.com", "williamhill.com",
]
ANALYST_MAX_TOKENS = 1200
MASTER_MAX_TOKENS = 1800

# if SELECTED_DATE > TODAY:
#     raise ValueError("SELECTED_DATE must be today or earlier")
print("Selected date:", SELECTED_DATE)

## 4. Create API clients

These objects only store configuration. No paid request happens in this cell.

In [ ]:
openrouter = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)
tavily = TavilyClient(api_key=TAVILY_API_KEY)
print("Clients ready")

## 5. Request the football-data.org response

Make one structured request, check the HTTP status, and inspect how many matches the provider returned.

In [ ]:
local_zone = ZoneInfo(MATCH_TIMEZONE)
utc_start = datetime.combine(SELECTED_DATE, time.min, tzinfo=local_zone).astimezone(timezone.utc)
utc_end = (datetime.combine(SELECTED_DATE, time.min, tzinfo=local_zone) + timedelta(days=1)).astimezone(timezone.utc)
print("Central match day in UTC:", utc_start, "to", utc_end)

fixture_response = httpx.get(
    f"{FOOTBALL_DATA_URL}/competitions/{FOOTBALL_DATA_COMPETITION}/matches",
    headers={"X-Auth-Token": FOOTBALL_DATA_API_KEY},
    params={
        "dateFrom": utc_start.date().isoformat(),
        "dateTo": utc_end.date().isoformat(),
    },
    timeout=90,
)
fixture_response.raise_for_status()
fixture_payload = fixture_response.json()
print("Provider matches returned:", len(fixture_payload.get("matches", [])))

## 6. Normalize the day's fixtures

Read the documented JSON fields directly and keep only kickoffs inside the exact Central-time match-day interval.

In [ ]:
fixtures = []
for item in fixture_payload.get("matches", []):
    kickoff = datetime.fromisoformat(item["utcDate"].replace("Z", "+00:00"))
    if not utc_start <= kickoff < utc_end:
        continue
    group_name = item.get("group") or item.get("stage") or ""
    fixtures.append({
        "match_key": f"football-data:{item['id']}",
        "match_date": SELECTED_DATE.isoformat(),
        "kickoff": kickoff.isoformat(),
        "kickoff_local": kickoff.astimezone(local_zone).isoformat(),
        "competition": item.get("competition", {}).get("name", "FIFA World Cup"),
        "group_name": group_name.replace("_", " " ).title() or None,
        "home_team": item["homeTeam"]["name"],
        "away_team": item["awayTeam"]["name"],
        "venue": item.get("venue"),
        "referees": [{
            "name": referee.get("name"),
            "role": referee.get("type"),
            "nationality": referee.get("nationality"),
        } for referee in item.get("referees", [])],
        "source": "football-data.org",
    })

print(json.dumps(fixtures, indent=2))

## 7. Choose one match

The notebook analyzes one match so you can understand the data and control cost. Change `MATCH_INDEX`, rerun from here, and repeat for another fixture. The production pipeline loops over all fixtures.

In [ ]:
if not fixtures:
    raise RuntimeError("No World Cup fixtures were found for the selected date")

MATCH_INDEX = -1
match = fixtures[MATCH_INDEX]
print(json.dumps(match, indent=2))

## 8. Research team news and form

This first Tavily query focuses on sporting evidence rather than odds. Inspect its answer and source URLs.

In [ ]:
display_date = SELECTED_DATE.strftime("%B %d, %Y")
fixture_label = f"{match['home_team']} vs {match['away_team']} FIFA World Cup 2026 {display_date}"
referee_names = ", ".join(r.get("name", "") for r in match.get("referees", []) if r.get("name"))
team_query = f"{fixture_label} latest team news injuries suspensions predicted lineups last 5 matches head to head"
print("Team-news query characters:", len(team_query))
team_research = tavily.search(
    query=team_query,
    search_depth=TAVILY_SEARCH_DEPTH,
    max_results=TEAM_NEWS_MAX_RESULTS,
    include_answer=False,
    exclude_domains=["instagram.com", "facebook.com", "x.com", "tiktok.com", "youtube.com"],
)
team_research = {"results": [{
    "title": result.get("title"), "url": result.get("url"),
    "content": (result.get("content") or "")[:TEAM_NEWS_CONTENT_CHARS],
} for result in team_research.get("results", [])]}
print("Retrieved results:")
for result in team_research.get("results", []):
    print("-", result.get("title"), result.get("url"))
    print((result.get("content") or "")[:500])

## 9. Research betting markets

This separate query asks for common 1X2, totals, and both-teams-to-score prices. Odds must retain their source and timestamp; search results may be stale or jurisdiction-specific.

In [ ]:
odds_query = (f"{fixture_label} latest bookmaker odds 1X2 double chance draw no bet Asian handicap "
              "totals over under 1.5 2.5 3.5 both teams to score")
print("Betting query characters:", len(odds_query))
odds_research = tavily.search(
    query=odds_query,
    search_depth=TAVILY_SEARCH_DEPTH,
    max_results=BETTING_MAX_RESULTS,
    include_answer=False,
    include_domains=BETTING_ALLOWED_DOMAINS,
    exclude_domains=["instagram.com", "facebook.com", "x.com", "tiktok.com", "youtube.com"],
)
odds_results = [result for result in odds_research.get("results", []) if any(
    (urlparse(result.get("url") or "").hostname or "").lower() == domain
    or (urlparse(result.get("url") or "").hostname or "").lower().endswith(f".{domain}")
    for domain in BETTING_ALLOWED_DOMAINS
)]
odds_research = {"results": [{
    "title": result.get("title"), "url": result.get("url"),
    "content": (result.get("content") or "")[:BETTING_CONTENT_CHARS],
} for result in odds_results]}
print("Retrieved results:")
for result in odds_research.get("results", []):
    print("-", result.get("title"), result.get("url"))
    print((result.get("content") or "")[:500])

## 10. Research tactics, venue, and weather

The third query captures context that can change a prediction but may not appear on an odds page.

In [ ]:
context_query = f"{fixture_label} tactical preview venue weather referee {referee_names}".strip()
print("Context query characters:", len(context_query))
context_research = tavily.search(
    query=context_query,
    search_depth=TAVILY_SEARCH_DEPTH,
    max_results=CONTEXT_MAX_RESULTS,
    include_answer=False,
    exclude_domains=["instagram.com", "facebook.com", "x.com", "tiktok.com", "youtube.com"],
)
context_research = {"results": [{
    "title": result.get("title"), "url": result.get("url"),
    "content": (result.get("content") or "")[:CONTEXT_CONTENT_CHARS],
} for result in context_research.get("results", [])]}
print("Retrieved results:")
for result in context_research.get("results", []):
    print("-", result.get("title"), result.get("url"))
    print((result.get("content") or "")[:500])

## 11. Combine and compress the evidence

The raw Tavily snippets remain available for audit. One inexpensive DeepSeek call compresses them into a Pydantic-validated digest that is reused by every analyst and the master.

In [ ]:
research = {
    "team_news_and_form": team_research,
    "odds_and_markets": odds_research,
    "tactics_venue_weather": context_research,
}

research_json = json.dumps(research, ensure_ascii=False)
print("Raw research characters:", len(research_json))

class EvidencePoint(BaseModel):
    model_config = ConfigDict(extra="forbid")
    claim: str = Field(min_length=1, max_length=160)
    source_url: str | None = Field(default=None, max_length=600)

class BettingPoint(BaseModel):
    model_config = ConfigDict(extra="forbid")
    market: str = Field(min_length=1, max_length=100)
    price_or_probability: str = Field(min_length=1, max_length=120)
    source_url: str | None = Field(default=None, max_length=600)
    caveat: str | None = Field(default=None, max_length=120)

DigestGap = Annotated[str, Field(min_length=1, max_length=160)]

class ResearchDigest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    team_news_form_h2h: list[EvidencePoint] = Field(default_factory=list, max_length=5)
    betting_markets: list[BettingPoint] = Field(default_factory=list, max_length=6)
    tactics_venue_weather: list[EvidencePoint] = Field(default_factory=list, max_length=3)
    conflicts_and_gaps: list[DigestGap] = Field(default_factory=list, max_length=4)

summary_instruction = f"""Summarize the search evidence for {match['home_team']} vs {match['away_team']}.
Return one JSON object matching the supplied schema. Use Simplified Chinese for claims.
Preserve source URLs, quoted odds/lines, and uncertainty. Do not infer missing current facts.
Capture available 1X2, double-chance, draw-no-bet, Asian handicap, BTTS, and totals 1.5/2.5/3.5.
Keep only decision-relevant evidence and list contradictions or missing markets.
Schema: {json.dumps(ResearchDigest.model_json_schema(), ensure_ascii=False)}
Evidence: {research_json}"""
print("Research summarizer prompt characters:", len(summary_instruction))

summary_response = openrouter.chat.completions.create(
    model=RESEARCH_SUMMARY_MODEL,
    messages=[{"role": "user", "content": summary_instruction}],
    temperature=0,
    max_tokens=RESEARCH_SUMMARY_MAX_TOKENS,
    response_format={"type": "json_object"},
).choices[0].message.content.strip()
if summary_response.startswith("```"):
    summary_response = summary_response.split("\n", 1)[1].rsplit("```", 1)[0].strip()
research_digest = ResearchDigest.model_validate_json(summary_response).model_dump(mode="json")
research_digest_json = json.dumps(research_digest, ensure_ascii=False)
print("Research digest characters:", len(research_digest_json))
print(research_digest_json)

## 12. Build the analyst instruction

Every model receives the same prompt. This makes their different conclusions easier to compare.

In [ ]:
analyst_instruction = f"""You are an independent football analyst in an AI championship.
Analyze {match['home_team']} vs {match['away_team']} in {match['competition']} on {match['match_date']}.
Fixture details: group={match.get('group_name')}, kickoff={match.get('kickoff')},
venue={match.get('venue')}, referees={json.dumps(match.get('referees', []), ensure_ascii=False)}.

Use the research digest as current evidence, but also add your own independent football and tactical analysis.
Clearly separate retrieved facts from your own reasoning. Never invent current injuries, lineups, or odds.
Flag stale or conflicting evidence and account for bookmaker margin.

Return concise Markdown in exactly this order:
## 快速结论
- 赛果：主胜 / 平局 / 客胜
- 预测比分：
- 主胜 / 平局 / 客胜概率：合计 100%
- 保守投注：
- 进取投注：没有价值时写“无”
- 信心：0-100

## 详细分析
### Tavily 证据
### 独立分析
### 主要风险
证据不足时推荐“不下注”。先给简短可执行结论，再给理由。
快速结论不超过 180 个中文字符；详细分析不超过 600 个中文字符。
Write the entire response in Simplified Chinese, including all headings and explanations.

Research digest:
{research_digest_json}"""

print("Analyst prompt characters:", len(analyst_instruction))
print(analyst_instruction[:2500], "...")

## 13. Full five-model panel (commented out)

The production-style full panel is preserved below for reference but commented out to avoid accidental cost while learning.

In [ ]:
# model_outputs = {}
# for model_config in MODELS:
#     model_id = model_config["id"]
#     model_slug = model_config["model"]
#     print("Calling", model_slug)
#     try:
#         response = openrouter.chat.completions.create(
#             model=model_slug,
#             messages=[{"role": "user", "content": analyst_instruction}],
#             temperature=0.2,
#             max_tokens=ANALYST_MAX_TOKENS,
#         )
#         model_outputs[model_id] = response.choices[0].message.content
#     except Exception as error:
#         model_outputs[model_id] = f"Model unavailable: {type(error).__name__}: {error}"

## 13A. Low-cost two-model test

This cell actually runs only DeepSeek V3 and GPT-4.1 Mini. It is the recommended notebook test path.

In [ ]:
TEST_MODELS = [
    {"id": "deepseek-v3", "model": "deepseek/deepseek-chat"},
    {"id": "gpt-4.1-mini", "model": "openai/gpt-4.1-mini"},
]
model_outputs = {}
for model_config in TEST_MODELS:
    print("Calling", model_config["model"], "with prompt characters:", len(analyst_instruction))
    response = openrouter.chat.completions.create(
        model=model_config["model"],
        messages=[{"role": "user", "content": analyst_instruction}],
        temperature=0.2,
        max_tokens=ANALYST_MAX_TOKENS,
    )
    model_outputs[model_config["id"]] = response.choices[0].message.content
print("Completed models:", list(model_outputs))

## 14. Read the independent opinions

Pause here and compare reasoning before asking the master model to synthesize it.

In [ ]:
for model_id, opinion in model_outputs.items():
    print("\n" + "=" * 80)
    print(model_id)
    print("=" * 80)
    print(opinion)

## 15. Build and run the master synthesis

The master receives both original research and panel outputs. It is told not to use simple majority voting and to recommend no bet when evidence is weak.

In [ ]:
master_instruction = f"""Act as chair of a football prediction panel.
Synthesize the reports instead of deciding by majority vote alone. Weigh source quality, reasoning,
market prices, team availability, and disagreement. Never invent facts or odds.

Return Markdown in exactly this order:
## 最终结论
- 赛果：
- 预测比分：
- 主胜 / 平局 / 客胜概率：合计 100%
- 保守投注：
- 进取投注：没有价值时写“无”
- 最终信心：0-100

## 综合分析
解释模型共识与分歧、最强 Tavily 证据、你的独立综合判断和失效风险。
最后添加简短理性投注提示；没有明确优势时推荐“不下注”。先给最终结论，再给分析。
最终结论不超过 220 个中文字符；综合分析不超过 800 个中文字符。
Write the entire final recommendation in Simplified Chinese.

Match: {json.dumps(match)}
Research digest: {research_digest_json}
Panel reports: {json.dumps(model_outputs, ensure_ascii=False)}"""

print("Master prompt characters:", len(master_instruction))
master_response = openrouter.chat.completions.create(
    model=MASTER_MODEL,
    messages=[{"role": "user", "content": master_instruction}],
    temperature=0.1,
    max_tokens=MASTER_MAX_TOKENS,
)
final_recommendation = master_response.choices[0].message.content
print(final_recommendation)

## 16. Create SQLite tables

SQLite keeps match identity separate from large JSON analysis payloads. The schema is created directly here so you can see every column.

In [ ]:
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
db = sqlite3.connect(DB_PATH)
db.executescript("""
CREATE TABLE IF NOT EXISTS matches (
    match_key TEXT PRIMARY KEY,
    match_date TEXT NOT NULL,
    kickoff TEXT,
    kickoff_local TEXT,
    competition TEXT,
    group_name TEXT,
    home_team TEXT,
    away_team TEXT,
    venue TEXT,
    referees_json TEXT NOT NULL DEFAULT '[]',
    source TEXT
);
CREATE INDEX IF NOT EXISTS idx_matches_date ON matches(match_date);
CREATE TABLE IF NOT EXISTS analyses (
    match_key TEXT PRIMARY KEY,
    research_json TEXT NOT NULL,
    model_outputs_json TEXT NOT NULL,
    final_output TEXT NOT NULL,
    analyzed_at TEXT DEFAULT CURRENT_TIMESTAMP
);
""")
existing_columns = {row[1] for row in db.execute("PRAGMA table_info(matches)")}
for column in ("kickoff_local", "group_name", "venue"):
    if column not in existing_columns:
        db.execute(f"ALTER TABLE matches ADD COLUMN {column} TEXT")
if "referees_json" not in existing_columns:
    db.execute("ALTER TABLE matches ADD COLUMN referees_json TEXT NOT NULL DEFAULT '[]'")
for legacy_column in ("status", "city"):
    if legacy_column in existing_columns:
        db.execute(f"ALTER TABLE matches DROP COLUMN {legacy_column}")
db.commit()
print("Database ready:", DB_PATH)

## 17. Save this match and analysis

`ON CONFLICT` means rerunning the notebook refreshes the same match rather than creating a duplicate.

In [ ]:
db.execute("""
INSERT INTO matches(match_key, match_date, kickoff, kickoff_local, competition, group_name, home_team, away_team, venue, referees_json, source)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(match_key) DO UPDATE SET
  match_date=excluded.match_date, kickoff=excluded.kickoff, kickoff_local=excluded.kickoff_local,
  competition=excluded.competition,
  group_name=excluded.group_name, home_team=excluded.home_team, away_team=excluded.away_team,
  venue=excluded.venue, referees_json=excluded.referees_json, source=excluded.source
""", (
    match["match_key"], match["match_date"], match.get("kickoff"), match.get("kickoff_local"),
    match["competition"],
    match.get("group_name"), match["home_team"], match["away_team"],
    match.get("venue"), json.dumps(match.get("referees", [])), match["source"],
))

db.execute("""
INSERT INTO analyses(match_key, research_json, model_outputs_json, final_output)
VALUES (?, ?, ?, ?)
ON CONFLICT(match_key) DO UPDATE SET
  research_json=excluded.research_json,
  model_outputs_json=excluded.model_outputs_json,
  final_output=excluded.final_output,
  analyzed_at=CURRENT_TIMESTAMP
""", (
    match["match_key"], json.dumps({"raw": research, "digest": research_digest}, ensure_ascii=False),
    json.dumps(model_outputs, ensure_ascii=False), final_recommendation,
))

db.commit()
print("Saved", match["match_key"])

## 18. Query history by date

This is the same basic operation the production Gradio app performs when its date picker changes.

In [ ]:
db.row_factory = sqlite3.Row
saved_rows = db.execute("""
SELECT m.*, a.model_outputs_json, a.final_output, a.analyzed_at
FROM matches m LEFT JOIN analyses a USING(match_key)
WHERE m.match_date = ?
ORDER BY m.kickoff
""", (SELECTED_DATE.isoformat(),)).fetchall()

print("Saved matches on this date:", len(saved_rows))
for row in saved_rows:
    print(row["home_team"], "vs", row["away_team"], "| analyzed:", row["analyzed_at"])

db.close()

## 19. Display the current result in Gradio

This final notebook cell is intentionally static: it displays the match you just analyzed without hiding data-loading logic in callbacks. Use `app.py` for the interactive historical date browser.

In [ ]:
with gr.Blocks(title="AI World Cup Championship") as demo:
    gr.Markdown("# AI World Cup Championship")
    gr.Markdown(
        f"## {match['home_team']} vs {match['away_team']}\n"
        f"**{match['competition']}** | {match.get('group_name') or 'Group TBD'} | "
        f"Central: {match.get('kickoff_local') or 'Kickoff TBD'} | UTC: {match.get('kickoff') or 'TBD'} | "
        f"{match.get('venue') or 'Venue TBD'}\n\n"
        f"**Referees:** {', '.join(r.get('name', '') for r in match.get('referees', [])) or 'TBD'}"
    )

    for start in range(0, len(TEST_MODELS), 3):
        with gr.Row():
            for model_config in TEST_MODELS[start:start + 3]:
                with gr.Column():
                    gr.Markdown(f"### {model_config['id']}")
                    opinion = model_outputs.get(model_config["id"], "暂无输出")
                    summary, marker, detail = opinion.partition("## 详细分析")
                    gr.Markdown(summary)
                    if marker:
                        with gr.Accordion("展开详细分析", open=False):
                            gr.Markdown(detail)

    gr.Markdown("## Master AI 最终结论")
    final_summary, final_marker, final_detail = final_recommendation.partition("## 综合分析")
    gr.Markdown(final_summary)
    if final_marker:
        with gr.Accordion("展开综合分析", open=False):
            gr.Markdown(final_detail)
    gr.Markdown("预测仅供参考。赔率会变化，任何结果都没有保证。")

demo.launch()